# Week 4 Lab 1 — CFD Data Production and Qualification

**Runtime:** CPU. **Estimated time:** 20–40 min including interpretation.  
This lab converts the Week-1 educational cavity solver into a controlled data generator. A completed run is a candidate field, not automatically an accepted label.


## Learning outcomes

- reproduce the common Re=100 benchmark;
- distinguish residual convergence from grid convergence;
- generate one assigned field using a fixed class protocol;
- export an NPZ field and a JSON quality card.


In [ ]:
import json, platform, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from cavity_project_utils import (CASE_MAP, GHIA, run_cavity, centerline_profiles,
                                  quality_gate, save_case)

STUDENT_ID = 1       # change to your assigned ID: 1,...,11
STUDENT_NAME = "Your Name"
assert STUDENT_ID in CASE_MAP
print(STUDENT_NAME, CASE_MAP[STUDENT_ID])
print("Python:", platform.python_version())


## Part 1 — Common Re=100 reproduction


In [ ]:
t0=time.time()
reproduction = run_cavity(Re=100, N=65, dt=1e-3, max_steps=12000,
                          poisson_iters=60, check_every=250,
                          min_steps=8000, tol=2e-6)
print(f"elapsed: {time.time()-t0:.1f} s")
gate100 = quality_gate(reproduction)
print(json.dumps(gate100, indent=2))


In [ ]:
def diagnostic_figure(result):
    uc,vc=centerline_profiles(result); Re=int(round(result["Re"]))
    fig,ax=plt.subplots(2,2,figsize=(10,8))
    ax[0,0].plot(uc,result["y"],label="CFD")
    if Re in GHIA:
        ref=GHIA[Re]; ax[0,0].scatter(ref["u"],ref["y"],s=20,label="Ghia")
    ax[0,0].set(xlabel="u/U at x/L=0.5",ylabel="y/L"); ax[0,0].legend(); ax[0,0].grid(alpha=.3)
    ax[0,1].plot(result["x"],vc,label="CFD")
    if Re in GHIA: ax[0,1].scatter(ref["x"],ref["v"],s=20,label="Ghia")
    ax[0,1].set(xlabel="x/L",ylabel="v/U at y/L=0.5"); ax[0,1].legend(); ax[0,1].grid(alpha=.3)
    ax[1,0].semilogy(result["residual_steps"],result["residual_values"],"o-")
    ax[1,0].set(xlabel="step",ylabel="relative vorticity-change residual"); ax[1,0].grid(alpha=.3)
    speed=np.hypot(result["u"],result["v"])
    ax[1,1].contourf(result["X"],result["Y"],speed,30)
    ax[1,1].streamplot(result["X"],result["Y"],result["u"],result["v"],density=1.1)
    ax[1,1].set(xlabel="x/L",ylabel="y/L",title=f"Re={Re}")
    fig.tight_layout(); return fig

diagnostic_figure(reproduction); plt.show()


In [ ]:
reproduction_path, card100 = save_case(reproduction, STUDENT_ID, "reproduction", "case_outputs")
print("saved", reproduction_path)


### Checkpoint 1

Record (E_u), (E_v), the final residual, ((x_v,y_v)), and (psi_{min}). Explain whether the residual is still decreasing, has plateaued, or satisfies the stopping rule. Do not proceed if fields contain non-finite values.


## Part 2 — Assigned production case


In [ ]:
cfg=CASE_MAP[STUDENT_ID]
t0=time.time()
production = run_cavity(Re=cfg["Re"], N=65, dt=1e-3, max_steps=20000,
                        poisson_iters=60, check_every=250,
                        min_steps=8000, tol=1e-5)
print(f"elapsed: {time.time()-t0:.1f} s")
gate=quality_gate(production)
print(json.dumps(gate,indent=2))
diagnostic_figure(production); plt.show()


In [ ]:
production_path, production_card = save_case(
    production, STUDENT_ID, cfg["split"], "case_outputs")
print("Submit:",production_path)
print("Submit:",production_path.with_name(production_path.stem+"_quality.json"))


## Written response

1. Is the field accepted by every quality check? Quote evidence.  
2. Compare vortex strength/location with Re=100.  
3. Distinguish iterative convergence, grid convergence, and ML generalization.  
4. Explain one mechanism by which CFD label error contaminates a surrogate.  
5. If your case is labeled `test`, explain why you may generate and inspect it but may not use it for model design.


## Article-output contract

<!-- MIE690A article-aligned validation v3 -->

**Role:** Foundational or supporting notebook; see ARTICLE_FIGURE_MAP.md for its evidence dependency.

All manuscript-facing figures must be generated from retained numerical/model outputs through the documented notebook or shared helper, saved under `results/`, and accompanied by machine-readable metrics. Do not redraw curves by eye or substitute a screenshot for a solver-to-reference comparison. The complete ownership table and exact output filenames are in [`ARTICLE_FIGURE_MAP.md`](../../ARTICLE_FIGURE_MAP.md).
